# Cygnus batch on a Google Colab **CPU-only** runtime

**What this is.** A driver for `python -m cygnus.batch` on a Colab CPU runtime, so large batches are
not limited by this workstation's disk (`docs/COLAB_HANDOFF.md`). It runs the pinned revision, runs
the offline test gate first, runs one batch with `--jobs 3`, compares the result with the committed
baseline leaf by leaf, and ships the evidence back to Drive.

**What it is not.** It is *not* the read-only single-product pilot
(`notebooks/cygnus_reanalysis_colab.ipynb`); it publishes nothing, submits nothing to any catalogue
or observer, deploys nothing, and changes no science threshold. It never selects a GPU or TPU
runtime: nothing in the pipeline needs an accelerator, and the compute units are better spent
elsewhere.

**Credentials.** No token, client secret, rclone config or password is entered here. Nothing in this
notebook prints or writes a credential, and no credential may appear in a saved output. The rclone
remote stays on the workstation — this notebook never calls rclone (it only *prints* the commands to
run there). Clear outputs before committing this notebook.

**Drive route: Option B**, `drive.mount()` inside Colab. The known risk is recorded in
`docs/COLAB_HANDOFF.md` problem 2: the workstation's `drive.file`-scoped rclone token only sees files
*rclone itself* created, so files this notebook writes through the mount may be invisible locally.
That is why step 4 writes a visibility probe and no result is trusted until a local listing finds it.

**The ledger is a companion, never a merge.** A Colab run writes its **own** `state/ledger.sqlite`.
It ships back as a named archived companion (`Cygnus/colab_runs/<batch>/ledger.sqlite` plus its
SHA-256, recorded in `docs/STATUS.md`) and is **never merged** into the local ledger. Committed
`campaigns/<id>/` records cite *local* ledger `run_id`s, so for this equivalence pilot the Colab
campaign outputs are compared and then **discarded**: they are never copied over the repository
records. Building a ledger import tool is deliberately out of scope.

**Resumable.** Every step is safe to re-run. The checkout is pinned and verified, the batch driver
skips campaigns that already finished (journal plus runner step reuse), and `state/` is restored
from the mount before the run — so a session that dies mid-batch can be resumed by running this
notebook from the top again. Re-running will not start the same campaign twice.

Read first: `docs/COLAB_HANDOFF.md` (problems 1-6) and `docs/AGENT_RUNBOOK.md` (*Batches*). Claim and
**commit** the batch's specs locally before running here, and do not run a local batch against the
same archives at the same time (there is no shared rate limiter).


In [ ]:
# Step 1. CPU-runtime check and the Drive mount (Option B).
import os
import shutil
import subprocess
from pathlib import Path

accelerator = shutil.which('nvidia-smi')
if (accelerator and subprocess.run([accelerator], capture_output=True).returncode == 0) \
        or os.environ.get('COLAB_TPU_ADDR'):
    raise RuntimeError('GPU/TPU runtime detected: choose Runtime > Change runtime type > CPU. This '
                       'work needs no accelerator and scoring it on one would be out of scope.')

from google.colab import drive                     # only present inside a Colab runtime
drive.mount('/content/drive')                      # Option B: the mount is how this runtime writes back

# SET_MOUNTED_CYGNUS_ROOT -- the mounted Cygnus root. No default is assumed: list
# /content/drive/MyDrive first and paste the directory that holds the Cygnus data (nothing outside
# Cygnus/ is read or written, ever).
CYGNUS_DRIVE_ROOT = Path('SET_MOUNTED_CYGNUS_ROOT')
if str(CYGNUS_DRIVE_ROOT).startswith('SET_'):
    raise ValueError('Set CYGNUS_DRIVE_ROOT to the mounted Cygnus directory before running anything')
if not CYGNUS_DRIVE_ROOT.is_dir():
    MYDRIVE = Path('/content/drive/MyDrive')
    if MYDRIVE.is_dir():
        names = sorted(p.name for p in MYDRIVE.iterdir())
        print('MyDrive listing (first 40):', names[:40])
        print('folder names containing "cygnus":',
              [name for name in names if 'cygnus' in name.lower()])
    raise FileNotFoundError(f'{CYGNUS_DRIVE_ROOT} is not a directory: the mount failed, or the '
                            'folder name is not the one this runtime sees (listing printed above)')
print('mounted Cygnus root:', CYGNUS_DRIVE_ROOT)

# Runtime size (Runtime > Change runtime type > CPU, High-RAM on). The work is CPU plus archive I/O:
# more cores allow more parallel campaigns; the cross-process throttle (cygnus.throttle) keeps each
# archive host at <= 2 request starts per second whatever the job count.
CPUS = os.cpu_count() or 2
try:
    RAM_GB = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1)
except (ValueError, OSError, AttributeError):
    RAM_GB = None
print(f'runtime: {CPUS} vCPU, {RAM_GB} GB RAM')


In [ ]:
# Step 2. Pinned checkout, install, and the offline test gate. Nothing runs before pytest passes.
import shutil
import shutil
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/mapsugui/ad_astra'   # public: cloning needs no credential
REPO_DIR = Path('/content/ad_astra')

# SET_PINNED_COMMIT -- optional override. Leave empty to use the reviewed pin (the repository HEAD
# recorded in docs/STATUS.md when this notebook was written); set a full 40-hex SHA to pin some other
# revision deliberately. The checkout is verified against the pin below, so a stale pin cannot pass.
SET_PINNED_COMMIT = ''
PINNED_COMMIT = SET_PINNED_COMMIT or '7a6ab05c0ace3d4205649db1772821cde3403d9d'
if len(PINNED_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in PINNED_COMMIT.lower()):
    raise ValueError('SET_PINNED_COMMIT: PINNED_COMMIT must be a full 40-hex commit SHA of ' + REPO_URL)

if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:                                    # re-entrant: fetch, keep the working tree and its batch state
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--quiet', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', '--detach', PINNED_COMMIT], check=True)
PINNED_HEAD = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True,
                             capture_output=True, text=True).stdout.strip()
if PINNED_HEAD != PINNED_COMMIT:
    raise RuntimeError(f'checkout is {PINNED_HEAD}, not the pinned {PINNED_COMMIT}: '
                       'refusing to run unrecorded code')
changed = subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], check=True,
                         capture_output=True, text=True).stdout.strip()
print('pinned commit:', PINNED_HEAD)
print('working tree:', 'clean' if not changed else f'{len(changed.splitlines())} path(s) written by a run')

# [test,mast] as docs/COLAB_HANDOFF.md names them, plus [campaign]: the batch runner's own steps
# import yaml/numpy/scipy, which the Colab base image does not guarantee. No science code is added.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e',
                f'{REPO_DIR}[test,mast,campaign]'], check=True)
# The editable install writes a .pth file that a kernel started before it never reads, so the later
# cells' `from cygnus... import` would fail (seen 2026-09-26); subprocesses are unaffected.
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))
UV = subprocess.run(['uv', '--version'], capture_output=True, text=True).stdout.strip() \
    if shutil.which('uv') else 'uv not installed'
RUNTIME = {'python': sys.version.split()[0], 'uv': UV, 'commit': PINNED_HEAD}
print('runtime:', RUNTIME)

# The offline gate (same marker selection as the workstation: no network, slow or replay tests).
gate = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=str(REPO_DIR))
if gate.returncode != 0:
    raise RuntimeError('the test suite failed on this runtime: fix the environment or re-pin. Running '
                       'science from an unverified checkout would make the record untrustworthy.')
print('pytest passed on', RUNTIME['python'])


In [ ]:
# Step 3. VM scratch, and resume: restore state/ if a previous session left it on the mount.
import json
import os
import shutil
from pathlib import Path
from cygnus.colab import sha256_file

# SET_BATCH_ID -- the batch id the specs were claimed into locally (state/batches/<id>/).
BATCH = 'SET_BATCH_ID'
if BATCH.startswith('SET_') or not BATCH.strip() or '/' in BATCH or '\\' in BATCH:
    raise ValueError('Set BATCH to the claimed batch id, e.g. b01 (a directory name, not a path)')

# MODE 'equivalence': re-run three committed campaigns and compare leaf by leaf (docs/COLAB_HANDOFF.md).
# MODE 'production': run NEW targets listed in a committed spec list (one campaigns/<id>.yaml per
# line; `python -m cygnus.batch claim` + a committed copy of specs.txt), ship full outputs to Drive.
MODE = 'SET_MODE'                          # 'equivalence' or 'production'
SET_SPEC_LIST = ''                         # production: repo-relative path of the committed spec list
if MODE not in ('equivalence', 'production'):
    raise ValueError("Set MODE to 'equivalence' or 'production'")
if MODE == 'equivalence':
    PILOT_CAMPAIGNS = ('toi-2666-01', 'toi-1301-02', 'toi-2003-01')
else:
    spec_list = REPO_DIR / SET_SPEC_LIST
    if not SET_SPEC_LIST or not spec_list.is_file():
        raise FileNotFoundError(f'production needs SET_SPEC_LIST, a committed spec list (got {SET_SPEC_LIST!r})')
    PILOT_CAMPAIGNS = tuple(Path(line.strip()).stem for line in spec_list.read_text(encoding='utf-8').splitlines()
                            if line.strip() and not line.startswith('#'))
    missing = [c for c in PILOT_CAMPAIGNS if not (REPO_DIR / 'campaigns' / f'{c}.yaml').is_file()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} listed spec(s) are not in the pinned commit, e.g. {missing[:3]}')
    # production is for NEW targets only: no listed campaign may already have a committed record, except
    # the draft placeholder (status 'draft', outcome 'not_run') committed with the spec so the gate passes
    import subprocess as _sp
    def _committed_status(c):
        r = _sp.run(['git', '-C', str(REPO_DIR), 'show', f'{PINNED_COMMIT}:campaigns/{c}/sky_record.json'],
                    capture_output=True, text=True)
        return json.loads(r.stdout).get('status') if r.returncode == 0 else None
    have = [c for c in PILOT_CAMPAIGNS if _committed_status(c) not in (None, 'draft')]
    if have:
        raise ValueError(f'{len(have)} listed target(s) already have committed records (e.g. {have[:3]}): '
                         'production mode is for new targets only')
JOBS = 3 if MODE == 'equivalence' else max(2, min(8, CPUS - 1))
print(f'mode {MODE}: {len(PILOT_CAMPAIGNS)} campaign(s), --jobs {JOBS}')

# VM-local, disposable, and NEVER the Drive mount: FITS reads over the mount are slow and the
# intermediates are bulky (`cygnus.config.scratch_dir()` honours CYGNUS_SCRATCH).
VM_SCRATCH_ROOT = '/content/scratch'         # a path on the Colab VM, never a mounted one
if VM_SCRATCH_ROOT.startswith(str(CYGNUS_DRIVE_ROOT)):
    raise ValueError('CYGNUS_SCRATCH would be inside the Drive mount: use a VM path such as '
                     '/content/scratch (slow reads, and bulky products would be shipped back)')
SCRATCH = Path(VM_SCRATCH_ROOT)
SCRATCH.mkdir(parents=True, exist_ok=True)

COLAB_RUNS = CYGNUS_DRIVE_ROOT / 'colab_runs' / BATCH     # what this run ships back (step 8)
COLAB_DIR = REPO_DIR / 'state' / 'colab' / BATCH          # reports produced on this VM
LEDGER = REPO_DIR / 'state' / 'ledger.sqlite'
BATCH_DIR = REPO_DIR / 'state' / 'batches' / BATCH
COLAB_DIR.mkdir(parents=True, exist_ok=True)
LEDGER.parent.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'CYGNUS_SCRATCH': str(SCRATCH), 'CYGNUS_LEDGER': str(LEDGER), 'CYGNUS_RATE_S': '0.5'}

restored: list[list] = []
if (COLAB_RUNS / 'ledger.sqlite').is_file():
    for suffix in ('', '-wal', '-shm'):          # a restored SQLite needs its sidecar files
        source = COLAB_RUNS / f'ledger.sqlite{suffix}'
        if source.is_file():
            shutil.copyfile(source, Path(str(LEDGER) + suffix))
            restored.append([f'ledger.sqlite{suffix}', source.stat().st_size, sha256_file(source)[:16]])
if (COLAB_RUNS / 'state' / 'batches' / BATCH).is_dir():
    shutil.copytree(COLAB_RUNS / 'state' / 'batches' / BATCH, BATCH_DIR, dirs_exist_ok=True)
    for path in sorted(BATCH_DIR.rglob('*')):
        if path.is_file():
            restored.append([path.relative_to(REPO_DIR).as_posix(), path.stat().st_size,
                             sha256_file(path)[:16]])
# Only the ledger and the batch journal are restored: committed campaigns/ records are never taken
# from the mount. A batch lock is never shipped (it lives beside the ledger, not under state/batches/),
# and this notebook never removes one: deleting a live lock (a batch still running on this VM) would
# let a second batch start, and a stale one from a dead runtime is taken over by the driver itself,
# which checks the recorded pid on the same host (src/cygnus/batch.py, LedgerLock.acquire).
print(f'restored {len(restored)} file(s) from {COLAB_RUNS}' if restored
      else 'no previous state on the mount: this is a fresh run')
for relative, size, digest in restored[:20]:
    print(f'  {relative}  {size} B  sha256 {digest}…')
if len(restored) > 20:
    print(f'  … {len(restored) - 20} more (all recorded in run_context.json)')

RUN_CONTEXT = {
    'batch': BATCH, 'pinned_commit': PINNED_HEAD, 'python': RUNTIME['python'], 'uv': RUNTIME['uv'],
    'scratch': SCRATCH.as_posix(), 'mode': MODE, 'cpus': CPUS, 'ram_gb': RAM_GB, 'jobs': JOBS,
    'campaigns': list(PILOT_CAMPAIGNS), 'restored_files': restored,
    'spec_sha256': {f'campaigns/{cid}.yaml': sha256_file(REPO_DIR / 'campaigns' / f'{cid}.yaml')
                    for cid in PILOT_CAMPAIGNS},
}
(COLAB_DIR / 'run_context.json').write_text(json.dumps(RUN_CONTEXT, indent=1, sort_keys=True),
                                            encoding='utf-8', newline='\n')
print('scratch:', SCRATCH.as_posix(), '| ledger:', LEDGER.as_posix())
print('batch dir:', BATCH_DIR.as_posix(), '| run context:', (COLAB_DIR / 'run_context.json').as_posix())


In [ ]:
# Step 4. Drive visibility probe -- Option B is unproven until a LOCAL listing finds this file.
# docs/COLAB_HANDOFF.md problem 2: the workstation token's drive.file scope only sees files rclone
# itself created, so anything drive.mount() writes may be invisible to it. This sentinel with a
# recorded SHA-256 and timestamp is the empirical test; the notebook cannot run it from here.
import datetime
from cygnus.colab import sha256_file

PROBE_DIR = COLAB_RUNS / '_probe'
PROBE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PROBE = PROBE_DIR / f'probe-{BATCH}-{PROBE_STAMP}.txt'
PROBE.write_text('cygnus colab drive-visibility probe\n'
                 f'batch: {BATCH}\npinned_commit: {PINNED_HEAD}\nwritten_utc: {PROBE_STAMP}\n',
                 encoding='utf-8')
PROBE_SHA256 = sha256_file(PROBE)
print('probe written :', PROBE.as_posix())
print('probe sha256  :', PROBE_SHA256)
print()
print('Verify on the workstation (the remote name is whatever `rclone listremotes` prints there;')
print('docs/COLAB_HANDOFF.md problem 1 records it changing between sessions):')
print(f'  rclone lsf <remote>:Cygnus/colab_runs/{BATCH}/_probe/')
print(f'  rclone cat <remote>:Cygnus/colab_runs/{BATCH}/_probe/{PROBE.name}')
print(f'  # the file must be listed, and its content must hash to {PROBE_SHA256}')
print()
print('If it is not listed, the return path is broken: STOP. The uploads in step 8 cannot be read')
print('locally. Options: ask the user, or switch to Option A (rclone inside Colab with its config')
print('supplied through Colab Secrets -- never printed, logged or committed). Never assume visibility.')
print('<remote> is whatever `python -m cygnus.storage where` resolves in that harness (docs/STORAGE.md).')
print('The probe is deliberately outside MANIFEST.sha256: it is a visibility test, not an artifact.')


In [ ]:
# Step 5. Archive health from THIS egress IP (Google's IPs may be treated differently from home).
import subprocess
import sys

ARCHIVES_CHECK = COLAB_DIR / 'archives_check.txt'
health = subprocess.run([sys.executable, '-m', 'cygnus.multi', 'archives', '--check', 'all'],
                        cwd=str(REPO_DIR), env=ENV, capture_output=True, text=True, timeout=3600)
ARCHIVES_CHECK.write_text(f'$ python -m cygnus.multi archives --check all\nexit {health.returncode}\n\n'
                          f'--- stdout ---\n{health.stdout}\n--- stderr ---\n{health.stderr}\n',
                          encoding='utf-8', newline='\n')
print(health.stdout[-4000:] if health.stdout else '(no stdout)')
if health.returncode == 0:
    print('archive health check: exit 0 (full output saved for the record)')
else:
    print(f'WARNING: archive health check exited {health.returncode}. Some services did not answer; '
          'the batch records that per campaign as inconclusive/not_tested and never as passed. Read '
          f'{ARCHIVES_CHECK.as_posix()} and expect retries.')


In [ ]:
# Step 6. Baseline FIRST (read-only), then the batch itself.
# `cygnus.batch run` overwrites campaigns/<id>/ inside the clone, so the committed records this run is
# compared against are extracted now, from the pinned commit, into /content/baseline. `git archive`
# reads objects only -- it never touches the index or the working tree, so the run cannot overwrite the
# baseline. The comparison therefore tests the pinned revision's own code and specs against the pinned
# revision's committed record; a dirty workstation worktree cannot influence it.
import json
import subprocess
import sys
import time
from pathlib import Path
from cygnus.colab import extract_git_archive

BASELINE = Path('/content/baseline') / PINNED_COMMIT[:12]
if MODE == 'equivalence':
    landed = extract_git_archive(REPO_DIR, PINNED_COMMIT,
                                 [f'campaigns/{cid}' for cid in PILOT_CAMPAIGNS], BASELINE)
    print(f'baseline: {len(landed)} file(s) from {PINNED_COMMIT[:12]} -> {BASELINE.as_posix()}')

# Campaigns run in chunks; after each chunk the ledger, the batch state, every finished campaign's
# outputs and its archive products are copied to the mount (COLAB_RUNS), so a disconnect loses at most
# one chunk. The driver is resumable: finished campaigns are skipped, so re-running this cell is safe.
CHUNK = 3 if MODE == 'equivalence' else 100
SKIP_NAMES = {'normalized_series.csv'}          # regenerable per-sector series: large, not shipped


def checkpoint(campaigns):
    import sqlite3
    COLAB_RUNS.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(LEDGER) as source, sqlite3.connect(COLAB_RUNS / 'ledger.sqlite') as target:
        source.backup(target)                      # a consistent snapshot, not a copy of a live file
    shutil.copytree(BATCH_DIR, COLAB_RUNS / 'state' / 'batches' / BATCH, dirs_exist_ok=True)
    if MODE != 'production':
        return
    for cid in campaigns:
        out = REPO_DIR / 'campaigns' / cid
        if (out / 'sky_record.json').is_file():
            shutil.copytree(out, COLAB_RUNS / 'campaigns' / cid, dirs_exist_ok=True,
                            ignore=lambda d, names: [n for n in names if n in SKIP_NAMES])
        products = SCRATCH / f'campaign_{cid}'
        if products.is_dir():
            shutil.copytree(products, COLAB_RUNS / 'products' / cid, dirs_exist_ok=True)


import shutil
BATCH_LOG = COLAB_DIR / 'batch_run.log'
started = time.time()
EXIT_CODE = 0
chunks = [PILOT_CAMPAIGNS[k:k + CHUNK] for k in range(0, len(PILOT_CAMPAIGNS), CHUNK)]
with BATCH_LOG.open('a', encoding='utf-8', newline='\n') as log:
    for n, chunk in enumerate(chunks, 1):
        BATCH_COMMAND = [sys.executable, '-m', 'cygnus.batch', 'run', '--batch', BATCH, '--jobs', str(JOBS),
                         '--spec', *[f'campaigns/{cid}.yaml' for cid in chunk]]
        print(f'--- chunk {n}/{len(chunks)} ({len(chunk)} campaigns, {round(time.time() - started)} s elapsed)', flush=True)
        log.write(f'$ {" ".join(BATCH_COMMAND[1:8])} ... ({len(chunk)} specs)\n')
        process = subprocess.Popen(BATCH_COMMAND, cwd=str(REPO_DIR), env=ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:             # stream it: a tab that shows work stays alive longer
            print(line, end='')
            log.write(line)
        code = process.wait()
        EXIT_CODE = EXIT_CODE or code
        checkpoint(chunk)
        print(f'checkpoint {n}/{len(chunks)} written to {COLAB_RUNS}', flush=True)
RUN_SECONDS = round(time.time() - started, 1)
RUN_CONTEXT.update({'batch_exit_code': EXIT_CODE, 'batch_seconds': RUN_SECONDS, 'chunks': len(chunks)})
(COLAB_DIR / 'run_context.json').write_text(json.dumps(RUN_CONTEXT, indent=1, sort_keys=True),
                                            encoding='utf-8', newline='\n')
print(f'batch exit code {EXIT_CODE} after {RUN_SECONDS} s (log: {BATCH_LOG.as_posix()})')

SUMMARY = BATCH_DIR / 'SUMMARY.md'
print(SUMMARY.read_text(encoding='utf-8') if SUMMARY.is_file()
      else 'no SUMMARY.md: the batch did not start (read the log above)')
if EXIT_CODE != 0:
    print()
    print('A non-zero exit is a result to ship, not a reason to lose the evidence: SUMMARY.md and the')
    print('journal name what failed, and this cell is safe to re-run. Read the comparison next.')


In [ ]:
# Step 7. Equivalence against the committed baseline: every differing leaf is classified.
# Nothing is waved through: science differences are listed for a decision, and bookkeeping
# differences (checksums, config hashes, run-state flags, paths, timestamps, reworded notes) are
# listed so they are read rather than assumed away.
import json
from cygnus.colab import compare_campaigns, render_report, sha256_file
if MODE != 'equivalence':
    print('production mode: no committed baseline exists for new targets; comparison skipped')
else:

    # A declared, recorded round-off tolerance. The 2026-09-26 pilot measured a maximum relative
    # numerical difference of 7.2e-10 between this runtime and the workstation for the pinned commit's
    # own code (identical code, a different BLAS/build); 1e-8 is ~14x that ceiling with margin. The
    # tolerance exists so identical code cannot fail equivalence over floating-point round-off: every
    # number it catches is still reported, with its exact relative difference, and a difference beyond
    # it is a science difference that stops the pilot.
    NUMERIC_RTOL = 1e-8

    # vetting.json / vetting/* are written by `python -m cygnus.campaign vet`, which `cygnus.batch run`
    # does not run. Excluding them is a stated decision, and the report lists the excluded files.
    NOT_RUN_BY_THE_BATCH = ('vetting.json', 'vetting/*')
    reports, sections = [], []
    for cid in PILOT_CAMPAIGNS:
        result = compare_campaigns(BASELINE / 'campaigns' / cid, REPO_DIR / 'campaigns' / cid,
                                   ignore_files=NOT_RUN_BY_THE_BATCH, numeric_rtol=NUMERIC_RTOL)
        reports.append(result)
        rendered = render_report(result, title=f'{cid}: committed baseline vs this run')
        sections.append(rendered)
        print(rendered)

    totals = {key: sum(r['counts'][key] for r in reports)
              for key in ('files_compared', 'leaves_compared', 'science', 'bookkeeping', 'numerical',
                          'ignored', 'ignored_files', 'files_only_in_baseline', 'files_only_in_candidate',
                          'not_compared', 'problems')}
    numerical_entries = [entry for r in reports for entry in r['numerical']]
    max_relative = max((float(entry['relative']) for entry in numerical_entries), default=0.0)

    # The runner may rewrite a spec. The pinned specs' SHA-256s were recorded in step 3, before the run;
    # recompute them now so a silent rewrite cannot pass undetected.
    spec_rewrites = {rel: {'before': digest, 'after': sha256_file(REPO_DIR / rel)}
                     for rel, digest in RUN_CONTEXT.get('spec_sha256', {}).items()
                     if sha256_file(REPO_DIR / rel) != digest}

    header = [
        f'# Equivalence pilot: batch {BATCH} on a Colab CPU runtime', '',
        f'- pinned commit: `{PINNED_COMMIT}`', f'- batch: `{BATCH}`',
        f'- batch exit code: {EXIT_CODE} ({RUN_SECONDS} s wall time)',
        f"- runtime: python {RUN_CONTEXT['python']}, {RUN_CONTEXT['uv']}",
        f"- compared: {totals['files_compared']} file(s), {totals['leaves_compared']} leaves",
        f"- declared numeric tolerance: {NUMERIC_RTOL!r}; within-tolerance numerical differences: "
        f"{totals['numerical']}",
        f"- largest relative numerical difference: {max_relative:.3e}; spec rewrites during the run: "
        f"{len(spec_rewrites)}",
        f"- science differences: {totals['science']}; bookkeeping differences: {totals['bookkeeping']}; "
        f"explicitly ignored: {totals['ignored']} (+{totals['ignored_files']} ignored file(s))",
        f"- files only in the baseline: {totals['files_only_in_baseline']}; only in this run: "
        f"{totals['files_only_in_candidate']}; not compared: {totals['not_compared']}; "
        f"problems: {totals['problems']}", '',
        'Files excluded from the leaf comparison: `vetting.json`, `vetting/*` (written by '
        '`python -m cygnus.campaign vet`, which a batch run does not do). Markdown reports '
        '(`REPORT.md`, `SEARCH_LOG.md`), generated light-curve series and images are listed as not '
        'compared: they are prose or derived products, not independent measurements.', '',
        'No difference here is waved through. Each science difference must be explained (with the reason '
        'recorded) or fixed and the run repeated before any larger batch is proposed. The bookkeeping '
        'differences are an expected class, but they still have to be read: checksums, configuration '
        'hashes and run-state flags live in that class. The numeric tolerance is a declared decision, '
        'named in this report and in every numerical entry, and every number it catches is still listed.',
        '', '---', '']
    EQUIVALENCE = COLAB_DIR / 'EQUIVALENCE.md'
    body = '\n\n'.join(sections)
    if spec_rewrites:
        body += '\n\n## Spec rewrites during the run (the pinned spec changed; explain before anything '
        body = body + 'else)\n\n' + json.dumps(spec_rewrites, indent=1, sort_keys=True)
    EQUIVALENCE.write_text('\n'.join(header) + body + '\n', encoding='utf-8', newline='\n')
    (COLAB_DIR / 'EQUIVALENCE.json').write_text(json.dumps(reports, indent=1, sort_keys=True),
                                                encoding='utf-8', newline='\n')
    print('=' * 72)
    print('verdict:', ', '.join(sorted({r['status'] for r in reports})),
          ('| numerical differences %d, largest relative %.3e' % (totals['numerical'], max_relative))
          if numerical_entries else '')
    print('science differences total:', totals['science'])
    print('report:', EQUIVALENCE.as_posix())
    if {r['status'] for r in reports} - {'identical', 'bookkeeping_only'}:
        print('NOT EQUIVALENT yet: explain every science difference above, or fix the code and re-run.')


In [ ]:
import json
# Step 8. Ship the evidence back: the companion ledger, the batch journal, the reports, a manifest.
# The committed campaigns/<id>/ records are deliberately NOT uploaded here: see the opt-in cell below.
import shutil
from cygnus.colab import build_manifest, sha256_file, verify_manifest, write_manifest

UPLOAD = [('ledger.sqlite', LEDGER), (f'state/batches/{BATCH}', BATCH_DIR),
          (f'state/colab/{BATCH}', COLAB_DIR)]
checkpoint(PILOT_CAMPAIGNS)          # final state, ledger snapshot and (production) outputs
COLAB_RUNS.mkdir(parents=True, exist_ok=True)
for relative, source in UPLOAD:
    destination = COLAB_RUNS / relative
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
    else:
        raise FileNotFoundError(f'nothing to upload for {relative}: {source.as_posix()} is missing')
    print('uploaded', relative)

SHIPPED = [relative for relative, _ in UPLOAD] + [d for d in ('campaigns', 'products') if (COLAB_RUNS / d).is_dir()]
manifest = build_manifest(COLAB_RUNS, SHIPPED)
MANIFEST = write_manifest(COLAB_RUNS / 'MANIFEST.sha256', manifest,
                          comment=f'cygnus batch {BATCH}; pinned commit {PINNED_COMMIT}; '
                                  f'{len(manifest)} file(s) written by the Colab runtime')

# Read every uploaded file back from the mount and recompute its SHA-256. This proves the mount round
# trip only; whether the workstation token can see these files is the probe in step 4, checked there.
problems = verify_manifest(MANIFEST, COLAB_RUNS, ignore_extra=['_probe/*'])
if problems:
    print('UPLOAD VERIFICATION FAILED -- do not trust these artifacts until every problem is gone:')
    for problem in problems:
        print('  -', problem)
else:
    print(f'upload read back and verified from the mount: {len(manifest)} file(s)')
print()
print('Record in docs/STATUS.md (the paths as the workstation sees them):')
print(f'  companion ledger : Cygnus/colab_runs/{BATCH}/ledger.sqlite')
print(f'  ledger sha256    : {sha256_file(LEDGER)}')
print(f'  manifest         : Cygnus/colab_runs/{BATCH}/MANIFEST.sha256')
print(f'  manifest sha256  : {sha256_file(MANIFEST)}')
print(f'  pinned commit    : {PINNED_COMMIT}   (batch {BATCH}, exit {EXIT_CODE}, {RUN_SECONDS} s)')
print()
print('Local checks (step 4 and this upload), run on the workstation:')
print(f'  rclone lsf <remote>:Cygnus/colab_runs/{BATCH}/_probe/')
print(f'  rclone lsf <remote>:Cygnus/colab_runs/{BATCH}/')
print('MANIFEST.sha256 does not cover _probe/ on purpose: the probe is a visibility test whose own')
print('SHA-256 is printed in step 4, not a shipped artifact.')

# Where this run lives, in the form every harness can read (docs/STORAGE.md): LOCATION.json beside the
# data, with the exact Drive folder ID (the mount's user.drive.id attribute) when Colab exposes it.
from cygnus.storage import Route, folder_id, make_entry, upsert
MOUNT_ROUTE = Route('path', str(CYGNUS_DRIVE_ROOT), 'Colab mount')
LOCATION = make_entry(f'colab_runs/{BATCH}', kind='colab_run', writer=MOUNT_ROUTE.label,
                      producer='notebooks/cygnus_batch_colab.ipynb', commit=PINNED_COMMIT,
                      manifest_sha256=sha256_file(MANIFEST), files=len(manifest),
                      drive_folder_id=folder_id(MOUNT_ROUTE, f'colab_runs/{BATCH}'),
                      notes=f'batch {BATCH}; exit {EXIT_CODE}; {RUN_SECONDS} s; companion ledger, never merged')
LOCATION['visible_via'] = {MOUNT_ROUTE.label: {'state': 'visible' if not problems else 'not_visible',
                                               'utc': LOCATION['created_utc'],
                                               'note': f'{len(manifest)} file(s) read back from the mount'}}
(COLAB_RUNS / 'LOCATION.json').write_text(json.dumps(LOCATION, indent=1, sort_keys=True) + '\n', encoding='utf-8')
upsert(LOCATION)          # into this clone's storage/locations.jsonl; the clone cannot push, so:
print()
print('Open it:', LOCATION['links']['folder'] or LOCATION['links']['search'])
print('Commit this line to storage/locations.jsonl from any harness with git write access')
print('(or run `python -m cygnus.storage add <downloaded LOCATION.json>` there):')
print(json.dumps(LOCATION, sort_keys=True, ensure_ascii=False))


In [ ]:
# Step 9. OPT-IN and OFF by default: copy this run's campaigns/<id>/ outputs into the shipped tree.
# Only for a batch whose targets are NEW (no committed record cites a local ledger run_id for them).
# For the equivalence pilot it MUST stay off: overwriting campaigns/<id>/ would break ledger
# referential integrity, because those records cite local ledger run_ids and state/ledger.sqlite is
# local and unmerged. There is deliberately no ledger import tool yet -- docs/COLAB_HANDOFF.md
# problem 4, option (b) -- so an imported run_id would resolve to nothing.
if MODE == 'production':
    print('production mode: outputs of these NEW targets were shipped by the checkpoints (campaigns/, products/);')
    print('step 3 refused any target that already had a committed record, so nothing here overwrites one.')
SET_OVERWRITE_COMMITTED_RECORDS = 'NO'   # replace with the literal token below to enable, nothing less
if SET_OVERWRITE_COMMITTED_RECORDS != 'YES_I_HAVE_READ_THE_WARNING_AND_THE_TARGETS_ARE_NEW':
    print('not enabled: campaigns/<id>/ outputs stay out of the upload, and NOTHING in the repository')
    print('is overwritten by this notebook. This is the correct setting for the equivalence pilot.')
else:
    import shutil
    from cygnus.colab import build_manifest, committed_files, write_manifest

    print('WARNING: uploading campaign outputs. On the workstation these files overwrite committed')
    print('records whose ledger run_ids are NOT in this ledger, so those records become')
    print('unresolvable. Do this only for new targets, and record the import decision in')
    print('docs/STATUS.md before copying anything in.')
    # git decides what is committable: tracked plus not-ignored. No hand-written list, so *.fits,
    # state/ and the generated campaigns/*/tic*/sector*/normalized_series.csv can never be swept in.
    selected = committed_files(REPO_DIR, [f'campaigns/{cid}' for cid in PILOT_CAMPAIGNS])
    for relative in selected:
        destination = COLAB_RUNS / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(REPO_DIR / relative, destination)
    print(f'copied {len(selected)} committed file(s): .gitignore itself decided, so archive')
    print('products, state/ entries and the generated per-sector series are not among them')
    manifest = build_manifest(COLAB_RUNS, [relative for relative, _ in UPLOAD]
                              + [f'campaigns/{cid}' for cid in PILOT_CAMPAIGNS])
    write_manifest(COLAB_RUNS / 'MANIFEST.sha256', manifest,
                   comment=f'cygnus batch {BATCH} INCLUDING campaign outputs; pinned {PINNED_COMMIT}')
    print('MANIFEST.sha256 rewritten to cover the campaign outputs too')


## Next: on the workstation (not in this notebook)

1. **Record where the run lives, then check who can see it.** Commit the JSON line step 8 printed to
   `storage/locations.jsonl` (or `python -m cygnus.storage add LOCATION.json`), then in each harness run
   `python -m cygnus.storage check colab_runs/<batch>`: it records `visible` / `not_visible` for that
   harness's route. A `drive.file`-scoped rclone token does **not** see mount-written folders (checked
   2026-09-26); download from a harness that does, or open the folder link from the index. Never
   assume visibility, and never re-upload a `colab_runs/` folder through rclone (a second folder with
   the same name would appear; docs/STORAGE.md, one writer per area).
2. **Download and verify every checksum** — an empty problem list is the only pass:
   ```bash
   rclone copy <remote>:Cygnus/colab_runs/<batch> ./colab_download --progress
   # from the worktree root:
   python -c "from cygnus.colab import verify_manifest as v; print(v('colab_download/MANIFEST.sha256', 'colab_download', ignore_extra=['_probe/*']))"
   ```
   Any problem string means the *copy* is broken, not the science: re-download before reading it.
3. **Do not copy `campaigns/<id>/` in.** The pilot compared values and discards the outputs; the
   committed records stay as they are, because they cite local ledger `run_id`s.
4. **Archive the companion ledger exactly as shipped** — `Cygnus/colab_runs/<batch>/ledger.sqlite`
   with the SHA-256 printed in step 8. A named archived companion, **never merged** into
   `state/ledger.sqlite`.
5. `python -m cygnus.batch status --batch <batch>` against the restored journal, then
   `python -m pytest -q`, then the review in `docs/AGENT_RUNBOOK.md`.
6. Read `EQUIVALENCE.md`: every science difference explained or fixed; every bookkeeping difference
   read. Only then propose a larger Colab batch, using the wall time and units actually measured here.
7. **Record in `docs/STATUS.md`** (and `DATA_SOURCES.md` / `AGENTS.md` where the handoff asks):
   - the remote name that actually exists locally (`rclone listremotes`), the route (Option B,
     `drive.mount()`) and whether the visibility probe passed;
   - the pinned commit, the batch id, the companion ledger path and SHA-256, the manifest SHA-256;
   - **the compute units actually used, read from Colab's Resources panel** and reported as observed —
     never estimated, extrapolated or quoted from a rate you did not see. If you did not look, write
     that you did not look;
   - the limits found: session length, idle disconnects, archive behaviour from Google's egress IPs,
     wall time per campaign, and anything that forced a retry.

Nothing here is published, submitted or deployed, and no credential is written into the notebook, its
outputs or a commit.
